# Colab: Analiza pravicnosti (Fairness)

Analiziramo pravicnost modela koristeci **Equalized Odds** metriku.

**Equalized Odds** zahteva da model ima jednake performanse (TPR i FPR) za sve demografske grupe:
- **TPR (True Positive Rate / Recall)** - procenat ispravno detektovanih malignih slucajeva
- **FPR (False Positive Rate)** - procenat pogresno klasifikovanih benignih slucajeva

Analiziramo pravicnost po:
1. **Polu** (muski/zenski)
2. **Starosnoj grupi** (<30, 30-44, 45-59, 60-74, 75+)

**Preduslovi:** Pokrenite notebookove za treniranje prvo!

## 1. Inicijalizacija

In [ ]:
import os, sys, json

if os.path.exists('/content/colab_paths.json'):
    paths = json.load(open('/content/colab_paths.json'))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    paths = json.load(open('/content/drive/MyDrive/melanoma_results/colab_paths.json'))

sys.path.insert(0, paths['repo_dir'])

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation import compute_metrics_at_best_threshold
from src.fairness import full_fairness_report
from src.visualization import plot_fairness_bars

## 2. Ucitavanje rezultata

Koristimo model sa boljim AUC za analizu pravicnosti.

In [ ]:
results_dir = paths['results_dir']

# Ucitaj oba modela
with open(os.path.join(results_dir, 'cnn_results.pkl'), 'rb') as f:
    cnn_res = pickle.load(f)
with open(os.path.join(results_dir, 'effnet_results.pkl'), 'rb') as f:
    effnet_res = pickle.load(f)

# Koristi model sa boljim AUC
if effnet_res['mean_auc'] >= cnn_res['mean_auc']:
    results = effnet_res
    model_name = 'EfficientNet B0'
else:
    results = cnn_res
    model_name = 'Custom CNN'

oof_labels = results['oof_labels']
oof_probs = results['oof_probs']
oof_df = results['oof_df']

_, best_threshold = compute_metrics_at_best_threshold(oof_labels, oof_probs)
print(f"Model: {model_name}")
print(f"AUC: {results['mean_auc']:.4f}")
print(f"Optimalni prag: {best_threshold:.4f}")
print(f"Broj uzoraka: {len(oof_labels)}")

## 3. Pravicnost po polu

Analiziramo da li model jednako dobro prepoznaje melanom kod muskih i zenskih pacijenata.

In [ ]:
report = full_fairness_report(oof_df, oof_labels, oof_probs, threshold=best_threshold)

if 'sex' in report:
    print("Equalized Odds po polu:")
    display(report['sex'])
    
    tpr_disp = report['sex'].attrs.get('tpr_disparity', 'N/A')
    fpr_disp = report['sex'].attrs.get('fpr_disparity', 'N/A')
    print(f"\nTPR disparitet: {tpr_disp:.4f}" if isinstance(tpr_disp, float) else f"\nTPR disparitet: {tpr_disp}")
    print(f"FPR disparitet: {fpr_disp:.4f}" if isinstance(fpr_disp, float) else f"FPR disparitet: {fpr_disp}")
    
    fig = plot_fairness_bars(report['sex'], title=f'Equalized Odds po polu ({model_name})')
    plt.show()
else:
    print("Kolona 'sex' nije dostupna u podacima.")

**Interpretacija:** Idealno, TPR i FPR bi trebalo da budu jednaki za obe grupe. Veliki disparitet ukazuje na to da model moze biti pristrastan prema jednom polu.

## 4. Pravicnost po starosnoj grupi

Melanom je cesci kod starijih pacijenata. Proveravamo da li model jednako dobro funkcionise za sve starosne grupe.

In [ ]:
if 'age_group' in report:
    print("Equalized Odds po starosnoj grupi:")
    display(report['age_group'])
    
    tpr_disp = report['age_group'].attrs.get('tpr_disparity', 'N/A')
    fpr_disp = report['age_group'].attrs.get('fpr_disparity', 'N/A')
    print(f"\nTPR disparitet: {tpr_disp:.4f}" if isinstance(tpr_disp, float) else f"\nTPR disparitet: {tpr_disp}")
    print(f"FPR disparitet: {fpr_disp:.4f}" if isinstance(fpr_disp, float) else f"FPR disparitet: {fpr_disp}")
    
    fig = plot_fairness_bars(report['age_group'], title=f'Equalized Odds po starosti ({model_name})')
    plt.show()
else:
    print("Kolona 'age_approx' nije dostupna u podacima.")

## 5. Sumarni pregled dispariteta

In [ ]:
print(f"=== Sumarni pregled dispariteta za {model_name} ===")
print(f"{'Grupa':<20} {'TPR disparitet':>15} {'FPR disparitet':>15}")
print('-' * 52)

for group_name, fair_df in report.items():
    tpr_d = fair_df.attrs.get('tpr_disparity', float('nan'))
    fpr_d = fair_df.attrs.get('fpr_disparity', float('nan'))
    tpr_str = f"{tpr_d:.4f}" if not np.isnan(tpr_d) else "N/A"
    fpr_str = f"{fpr_d:.4f}" if not np.isnan(fpr_d) else "N/A"
    print(f"{group_name:<20} {tpr_str:>15} {fpr_str:>15}")

print()
print("Manji disparitet = pravicniji model (idealno = 0.0)")

## Zakljucak

Analiza pravicnosti nam pokazuje koliko model jednako funkcionise za razlicite demografske grupe.

**Kljucni nalazi:**
- Disparitet u TPR ukazuje na razliku u sposobnosti detekcije melanoma izmedju grupa
- Disparitet u FPR ukazuje na razliku u broju lazno pozitivnih rezultata
- Za medicinsku primenu, posebno je vazno da TPR bude visok i ujednacen za SVE grupe

**Moguca poboljsanja:**
- Veci i raznovrsniji dataset (posebno za nedovoljno zastupljene grupe)
- Augmentacija podataka fokusirana na manje zastupljene grupe
- Fairness-aware tehnike treniranja (npr. adversarial debiasing)
- Kalibracija modela po grupama